# 📧 Email Triage OpenEnv — Colab Setup

Run this notebook to:
1. Install all dependencies
2. Clone / upload the environment
3. Start the FastAPI server
4. Run baseline inference
5. Deploy to HuggingFace Spaces

**Free Colab tier is sufficient.**

## 1. Install Dependencies

In [ ]:
# Install all required packages
!pip install -q fastapi uvicorn pydantic openai pyyaml httpx python-multipart huggingface_hub[cli] pytest
print('✓ All packages installed')

## 2. Clone the Repository

In [ ]:
import os

# Option A: Clone from GitHub (replace with your repo URL after pushing)
GITHUB_REPO = "https://github.com/YOUR_USERNAME/email-triage-env"  # ← change this

if GITHUB_REPO != "https://github.com/YOUR_USERNAME/email-triage-env":
    !git clone {GITHUB_REPO} email-triage-env
    os.chdir('email-triage-env')
    print('✓ Cloned from GitHub')
else:
    print('⚠ No GitHub repo set. Upload the zip file using the cell below instead.')

In [ ]:
# Option B: Upload the zip file (run if you skipped GitHub clone)
from google.colab import files
import zipfile, os

# Uncomment to upload:
# uploaded = files.upload()  # upload email-triage-env.zip
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name) as z:
#     z.extractall('.')
# os.chdir('email-triage-env')
# print(f'✓ Extracted {zip_name}')

In [ ]:
# Verify project structure
import os
for root, dirs, files in os.walk('.'):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', '.git', '.venv']]
    level = root.replace('.', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        print(f'{indent}  {f}')

## 3. Configure API Keys

In [ ]:
import os
from getpass import getpass

# Set your API credentials
os.environ['HF_TOKEN'] = getpass('Enter HuggingFace token (or OpenAI key): ')
os.environ['OPENAI_API_KEY'] = os.environ['HF_TOKEN']

# Choose your LLM
os.environ['API_BASE_URL'] = 'https://api.openai.com/v1'  # or HF Inference API
os.environ['MODEL_NAME'] = 'gpt-4o-mini'                  # or any OpenAI-compat model

print('✓ API keys configured')
print(f'  Model: {os.environ["MODEL_NAME"]}')
print(f'  API:   {os.environ["API_BASE_URL"]}')

## 4. Run Tests

In [ ]:
!pytest tests/test_env.py -v --tb=short

## 5. Start FastAPI Server (background)

In [ ]:
import subprocess, time, requests

# Start the server in background
server_proc = subprocess.Popen(
    ['uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '7860'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

# Health check
try:
    r = requests.get('http://localhost:7860/')
    print('✓ Server running:', r.json()['name'])
except Exception as e:
    print('✗ Server failed to start:', e)
    print(server_proc.stderr.read().decode())

In [ ]:
import requests, json

BASE = 'http://localhost:7860'

# Reset environment
obs = requests.post(f'{BASE}/reset', json={'task_id': 'easy_triage'}).json()
print(f'Task: {obs["task_id"]} | Emails: {obs["pending_count"]} pending')

# Classify first email
result = requests.post(f'{BASE}/step', json={
    'action_type': 'classify',
    'email_id': 'e001',
    'urgency': 'critical',
    'category': 'technical_support'
}).json()
print(f'Step reward: {result["reward"]["step_reward"]}')
print(f'Result: {result["observation"]["last_action_result"]}')

# Validate
v = requests.get(f'{BASE}/validate').json()
print(f'\nOpenEnv validation: {v["status"]}')
print(json.dumps(v['checks'], indent=2))

## 6. Run Baseline Inference

In [ ]:
# Run baseline agent on all 3 tasks
# This uses the OpenAI client against your configured API
!python inference.py

In [ ]:
# Read and display scores
import json
try:
    with open('baseline_scores.json') as f:
        scores = json.load(f)
    print('=== BASELINE SCORES ===')
    for task, score in scores['scores'].items():
        bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
        print(f'  {task:20s}: {bar} {score:.4f}')
    avg = sum(scores['scores'].values()) / len(scores['scores'])
    print(f'  {"AVERAGE":20s}: {avg:.4f}')
    print(f'  Model: {scores["model"]}')
except FileNotFoundError:
    print('Run inference.py first')

## 7. Deploy to HuggingFace Spaces

In [ ]:
from huggingface_hub import HfApi, create_repo
import os

HF_TOKEN = os.environ.get('HF_TOKEN', '')
api = HfApi(token=HF_TOKEN)

# Get username
user_info = api.whoami()
HF_USERNAME = user_info['name']
SPACE_ID = f'{HF_USERNAME}/email-triage-env'

print(f'Deploying to: {SPACE_ID}')

# Create space
create_repo(
    repo_id=SPACE_ID,
    repo_type='space',
    space_sdk='docker',
    private=False,
    exist_ok=True,
    token=HF_TOKEN,
)
print('✓ Space created/verified')

In [ ]:
# Upload all files
FILES = [
    'Dockerfile', 'requirements.txt', 'server.py',
    'inference.py', 'openenv.yaml', 'README.md',
    'env/__init__.py', 'env/data.py',
    'env/environment.py', 'env/graders.py',
]

for path in FILES:
    if os.path.exists(path):
        api.upload_file(
            path_or_fileobj=path, path_in_repo=path,
            repo_id=SPACE_ID, repo_type='space', token=HF_TOKEN,
            commit_message=f'Upload {path}',
        )
        print(f'  ✓ {path}')
    else:
        print(f'  ✗ Missing: {path}')

print(f'\n✓ Deployed! View at: https://huggingface.co/spaces/{SPACE_ID}')

In [ ]:
# Set space secrets
secrets = {
    'HF_TOKEN': os.environ.get('HF_TOKEN', ''),
    'OPENAI_API_KEY': os.environ.get('OPENAI_API_KEY', ''),
    'API_BASE_URL': os.environ.get('API_BASE_URL', 'https://api.openai.com/v1'),
    'MODEL_NAME': os.environ.get('MODEL_NAME', 'gpt-4o-mini'),
}
for key, val in secrets.items():
    if val:
        api.add_space_secret(repo_id=SPACE_ID, key=key, value=val, token=HF_TOKEN)
        print(f'  ✓ Secret: {key}')

print(f'\nSpace URL: https://huggingface.co/spaces/{SPACE_ID}')
print('Note: Space may take 2-3 minutes to build and start.')